<a href="https://colab.research.google.com/github/ever1318-cpu/Apartment_Defect_AI/blob/main/ai_checker_progressive_pipeline_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Checker — 점진적 학습 파이프라인

클래스당 샘플 수를 단계별로 늘려가며 전체 파이프라인을 검증합니다.

## 스테이지 전략

| Stage | 샘플/클래스 | Epochs | 소요 시간 | 목적 |
|-------|-----------|--------|----------|------|
| **1** | 100장 | 3 | ~15분 | 초고속 스모크 테스트 |
| **2** | 300장 | 5 | ~30분 | 전체 파이프라인 빠른 검증 |
| **3** | 1,500장 | 10 | ~2시간 | 실용 수준 정확도 확인 |
| **4** | 5,000장 | 15 | ~8시간 | 운영 수준 품질 |
| **5** | 전체 | 20 | ~하루 | 최종 배포 모델 |

## 셀 실행 순서

| 셀 | 내용 | 이어받기 | 비고 |
|----|------|----------|------|
| 00 | GPU + Drive + 패키지 | - | 처음 1회 |
| 01 | **STAGE 선택** | - | ✏️ 여기서 1~5 선택 |
| 02 | DB 조회 + metadata | ✅ Drive 캐시 | 비밀번호 런타임 입력 |
| 03 | 스테이지별 샘플링 | - | 클래스당 N개 제한 |
| 04 | S3 → Drive 다운로드 | ✅ dl_progress | 한 번만 다운로드 |
| 05 | 5계층 모델 + 가중치 | - | - |
| 06 | 학습 루프 | ✅ last.pth | Early Stopping |
| 07 | 테스트 평가 + 비교 리포트 | - | 스테이지간 비교 |
| 08 | ONNX + TFLite 변환 | - | 온디바이스 배포 |

## Drive 저장 구조
```
AI_Checker_Final/
├── metadata.jsonl          ← 공통 (스테이지 무관)
├── taxonomy.json
├── dl_progress.json
├── ds_images/              ← 공통 이미지 (한 번만 다운로드)
├── stage_comparison.json   ← 스테이지별 성능 비교
├── stage_1/checkpoints/ + export/
├── stage_2/checkpoints/ + export/
├── stage_3/checkpoints/ + export/
└── stage_4/checkpoints/ + export/
```

## 런타임 재시작 후
```
셀 00 → 셀 01(STAGE 설정) → 셀 05 → 셀 06 (자동 이어받기)
```

> ⚠️ **2026-08 수정**: element_group/element_code/symptom_code 테이블은 실제 DB에 없음이 확인되어, part_detail 텍스트 → dict 매핑 방식으로 변경되었습니다.

## 셀 00 — GPU 확인 + Drive 마운트 + 패키지

In [3]:
!nvidia-smi
!pip install -q psycopg2-binary timm torch torchvision tqdm onnx onnxruntime

from google.colab import drive
drive.mount('/content/drive')

import os, json, re, time, shutil, hashlib, datetime, urllib.request, getpass, random
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import torch, torch.nn as nn, timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

print(f'✅ 준비 완료 | PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')

Thu Aug  6 13:27:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 셀 01 — 설정 (여기만 수정)

In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️  STAGE 설정 — 원하는 단계를 선택하세요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# STAGE 1:   100장/클래스 → 초고속 파이프라인 스모크 테스트 (~15분)
# STAGE 2:   300장/클래스 → 전체 파이프라인 빠른 검증        (~30분)
# STAGE 3: 1,500장/클래스 → 실용 수준 정확도 확인            (~2시간)
# STAGE 4: 5,000장/클래스 → 운영 수준 품질                   (~8시간)
# STAGE 5: 전체 데이터    → 최종 배포 모델                    (~하루)

STAGE = 1   # ← 여기서 1~5 선택

STAGE_CONFIG = {
    1: {'samples_per_class':   100, 'epochs': 3,  'batch': 32,
        'desc': '스모크 테스트 (15분)'},
    2: {'samples_per_class':   300, 'epochs': 5,  'batch': 32,
        'desc': '빠른 검증 (30분)'},
    3: {'samples_per_class': 1_500, 'epochs': 10, 'batch': 32,
        'desc': '실용 수준 (2시간)'},
    4: {'samples_per_class': 5_000, 'epochs': 15, 'batch': 64,
        'desc': '운영 수준 (8시간)'},
    5: {'samples_per_class':  None, 'epochs': 20, 'batch': 64,
        'desc': '최종 배포 (전체 데이터)'},
}

cfg = STAGE_CONFIG[STAGE]
SAMPLES_PER_CLASS = cfg['samples_per_class']   # None = 제한 없음
EPOCHS            = cfg['epochs']
BATCH             = cfg['batch']

# ── 공통 설정 ────────────────────────────────────
BK      = '/content/drive/MyDrive/AI_Checker_Final'
IMG_DIR = Path(f'{BK}/ds_images')   # Drive 영구 저장

DB_HOST = 'w-backupdb.c9u0e882q8zp.ap-northeast-2.rds.amazonaws.com'
DB_PORT = 5432
DB_NAME = 'BackupDB'
DB_USER = 'TruePostgres'
DB_PASS = ''   # 셀 02 실행 시 입력창

S3_REGION = 'ap-northeast-2'
S3_BUCKET = 'wmcsm-defect-file'

TYPES     = ['QUALITY']
TRAIN_MIN = 50      # 전체 DB 기준 최소 이미지 수 (클래스 포함 여부 판단)
SEED      = 42
LR        = 3e-4
MODEL_N   = 'convnext_tiny'
IMGSZ     = 224
PATIENCE  = 3       # Early Stopping
BATCH_DL  = 300     # 다운로드 체크포인트 간격

HEAD_LOSS_W = {
    'element_group': 1.0,
    'element_code':  1.5,
    'part_detail':   2.0,
    'symptom_code':  2.0,
    'cause':         1.5,
}
MINORITY_BOOST = 2.0

# ── 스테이지별 저장 경로 ─────────────────────────
STAGE_DIR = Path(f'{BK}/stage_{STAGE}')
CKPT_DIR  = STAGE_DIR / 'checkpoints'
EXPORT_DIR = STAGE_DIR / 'export'
META_PATH = Path(f'{BK}/metadata.jsonl')    # 공통 (스테이지 무관)
TAX_PATH  = Path(f'{BK}/taxonomy.json')
DL_PROG   = Path(f'{BK}/dl_progress.json')  # 공통
BEST_PATH = CKPT_DIR / 'best.pth'
LAST_PATH = CKPT_DIR / 'last.pth'
LOG_PATH  = CKPT_DIR / 'train_log.json'

for p in [BK, str(IMG_DIR), str(CKPT_DIR), str(EXPORT_DIR)]:
    os.makedirs(p, exist_ok=True)

print(f'\n{"="*52}')
print(f'  STAGE {STAGE} — {cfg["desc"]}')
print(f'{"="*52}')
lbl = f'{SAMPLES_PER_CLASS:,}장/클래스' if SAMPLES_PER_CLASS else '전체 데이터'
print(f'  샘플 제한  : {lbl}')
print(f'  Epochs    : {EPOCHS}')
print(f'  Batch     : {BATCH}')
print(f'  저장 경로  : {STAGE_DIR}')
print(f'{"="*52}')


  STAGE 1 — 스모크 테스트 (15분)
  샘플 제한  : 100장/클래스
  Epochs    : 3
  Batch     : 32
  저장 경로  : /content/drive/MyDrive/AI_Checker_Final/stage_1


## 셀 02 — DB 조회 + metadata 생성 (Drive 캐시 우선)

In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 중요: element_group/element_code/symptom_code 테이블은 DB에 없음
#       (2026-08 확인). 아래는 실제 존재하는 컬럼만 사용합니다.
#
#   L1 element_group : part_detail.name → Python dict 매핑
#   L2 element_code   : part_detail.name → Python dict 매핑
#   L3 part_detail    : defect_part_detail.name (DB 원본, ~68종)
#   L4 symptom_code   : defect_description/remark 텍스트 → 키워드 추출
#   L5 cause          : defect_cause.name (DB 원본: 시공불량/미시공/흠집/오염/파손 등)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import psycopg2

def assign_split(did, seed=42):
    h = int(hashlib.md5(f'{seed}_{did}'.encode()).hexdigest(), 16) % 1000
    return 'train' if h < 800 else ('val' if h < 900 else 'test')

def norm_phase(v):
    u = (v or '').upper()
    return 'BEFORE' if 'BEFORE' in u else ('AFTER' if 'AFTER' in u else u or 'ETC')

def s3_url(path):
    if str(path).startswith('http'): return str(path)
    key = str(path).split('/', 3)[3] if str(path).startswith('s3://') else str(path).lstrip('/')
    return f'https://{S3_BUCKET}.s3.{S3_REGION}.amazonaws.com/{key}'

safe = lambda s: re.sub(r'[\\/:*?"<>|]+', '_', str(s).strip())

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# part_detail.name(한글) → element_code 매핑 딕셔너리
# (하자분류체계_a_부재코드정의서_v0.1 기준, 부분일치로 매칭)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ELEMENT_MAP = [
    # (검색 키워드들, element_code)
    (['벽지','도배'],              'FIN.WLP'),
    (['도장','페인트'],             'FIN.PNT'),
    (['타일','줄눈'],               'FIN.TIL'),
    (['마루','강마루','장판','바닥재'], 'FIN.FLR'),
    (['몰딩','걸레받이','베이스'],    'FIN.MLD'),
    (['실리콘','코킹'],             'FIN.SIL'),
    (['천장재','천정재','우물천장','등박스'], 'FIN.CLG'),
    (['석재','대리석','계단석'],      'FIN.STN'),
    (['창문','유리','거실창','발코니창'], 'WIN.WDW'),
    (['창틀','문선'],               'WIN.FRM'),
    (['레일','롤러','크리센트','방충망'], 'WIN.HDW'),
    (['문짝','현관도어','방문','욕실문'], 'DOR.DFL'),
    (['문틀','문상방','문지방'],      'DOR.DFR'),
    (['경첩','손잡이','도어락','스토퍼'], 'DOR.DHW'),
    (['중문','슬라이딩','미닫이'],     'DOR.SLD'),
    (['싱크대','상부장','하부장'],    'FUR.SNK'),
    (['붙박이장','옷걸이봉'],         'FUR.WDR'),
    (['선반','수납','팬트리','신발장'], 'FUR.SHF'),
    (['상판','카운터','아일랜드'],    'FUR.CNT'),
    (['변기','도기'],               'SAN.TOI'),
    (['세면대','세면기'],            'SAN.SIN'),
    (['욕조','샤워'],               'SAN.BTH'),
    (['수건걸이','휴지걸이','거울'],  'SAN.ACC'),
    (['스위치','조광기'],            'MEP.SWT'),
    (['콘센트'],                   'MEP.OUT'),
    (['조명','다운라이트','팬던트','센서등'], 'MEP.LGT'),
    (['수전'],                     'MEP.FCT'),
    (['배수','트랩','하수구'],       'MEP.DRN'),
    (['환기','급기','실외기'],       'MEP.HVC'),
    (['난방','온도조절기','분배기'], 'MEP.HTG'),
    (['인터폰','월패드','인터넷단자'], 'MEP.INT'),
    (['콘크리트','슬래브','기둥'],   'STR.CON'),
    (['방수'],                     'STR.WPF'),
    (['외벽','외장재','파라펫','드라이비트'], 'STR.EXT'),
    (['발코니','난간'],             'STR.BLC'),
]

def map_element_code(part_detail_name: str) -> str:
    """part_detail 텍스트를 element_code로 매핑. 매칭 안 되면 ETC.OTH"""
    if not part_detail_name:
        return 'ETC.OTH'
    name = part_detail_name.strip()
    for keywords, code in ELEMENT_MAP:
        if any(kw in name for kw in keywords):
            return code
    return 'ETC.OTH'

def element_group_of(element_code: str) -> str:
    return element_code.split('.')[0]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 텍스트(defect_description/remark) → symptom_code 키워드 추출
# (하자분류체계_현상코드정의서_v0.1 기준)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SYMPTOM_MAP = [
    (['균열','크랙','갈라짐'],        'CRK'),
    (['스크래치','긁힘','스크레치'],   'SCR'),
    (['찍힘','눌림'],                'DNT'),
    (['파손','깨짐','깨진'],          'CHP'),
    (['구멍','천공'],                'HOL'),
    (['이격','틈','벌어짐'],          'GAP'),
    (['들뜸','부풀음','부풂'],        'LFT'),
    (['박리','벗겨짐','벗겨진'],       'PEL'),
    (['어긋남','단차'],               'MIS'),
    (['분리','떨어짐'],               'SEP'),
    (['오염','얼룩'],                'STN'),
    (['먼지','이물'],                'DST'),
    (['페인트 묻음','페인트묻음'],     'PNS'),
    (['녹','부식'],                  'RST'),
    (['곰팡이','결로'],               'MLD'),
    (['누수','누출','물샘'],          'LEK'),
    (['소음','소리'],                'NOI'),
    (['작동불량','작동 불량','고장'], 'MOV'),
    (['수평','수직','기울어짐','기울음'], 'SLP'),
    (['고정불량','흔들림','헐거움'],   'FIX'),
    (['변색'],                      'DIS'),
    (['색상 불균일','얼룩덜룩'],       'UNV'),
    (['톤 차이','톤차이'],            'TON'),
    (['마감불량','마감 불량'],         'FIN_SYM'),
    (['불균일','평활도'],             'UNE'),
    (['모서리'],                    'EDG'),
]

def map_symptom_code(*texts) -> str:
    """defect_description, remark 등 텍스트에서 키워드로 현상코드 추출"""
    combined = ' '.join(t for t in texts if t)
    if not combined:
        return '기타'
    for keywords, code in SYMPTOM_MAP:
        if any(kw in combined for kw in keywords):
            return code
    return '기타'

# ── Drive 캐시 확인 ──────────────────────────────────────────
if META_PATH.exists() and TAX_PATH.exists():
    print('✅ Drive 캐시 로드 (DB 재조회 생략)')
    recs = [json.loads(l) for l in META_PATH.open(encoding='utf-8')]
    tax  = json.load(TAX_PATH.open(encoding='utf-8'))
else:
    if not DB_PASS:
        DB_PASS = getpass.getpass('🔑 DB 비밀번호: ')

    print('[1/5] DB 조회 중 (확인된 실제 테이블만 사용)...')
    conn = psycopg2.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
        user=DB_USER, password=DB_PASS, connect_timeout=10)
    cur = conn.cursor()

    # ✅ 실제 존재 확인된 테이블/컬럼만 사용
    cur.execute("""
        SELECT
            d.id,
            f.full_path,
            f.bucket_name,
            f.file_type,
            pd.name    AS part_detail,
            wk.name    AS gongjong,
            cz.name    AS cause,
            dg.name    AS area,
            s.site_code,
            d.defect_description,
            d.remark,
            d.worker_remark,
            d.result_remark
        FROM site_defect d
        JOIN site_defect_file f          ON f.defect_id = d.id
        JOIN site_site_defect_item i     ON i.id = d.site_defect_item_id
        JOIN site_site_part_detail_map m ON m.id = i.site_site_part_detail_map_id
        JOIN defect_part_detail pd       ON pd.id = m.part_detail_id
        LEFT JOIN defect_work_kind wk    ON wk.id = d.work_kind_id
        LEFT JOIN defect_cause cz        ON cz.id = d.cause_id
        JOIN defect_ho h2   ON h2.id = d.ho_id
        JOIN defect_dong dg ON dg.id = h2.dong_id
        JOIN defect_site s  ON s.id  = dg.site_id
        WHERE d.is_deleted = false
          AND d.type = ANY(%s)
          AND pd.name IS NOT NULL
          AND f.full_path IS NOT NULL
    """, (TYPES,))
    cols = [c[0] for c in cur.description]
    raw  = [dict(zip(cols, row)) for row in cur.fetchall()]
    conn.close()
    print(f'  {len(raw):,}행 조회 완료')

    # ── 5계층 라벨 부여 (Python 매핑) ─────────────────────────
    print('[2/5] 5계층 라벨 생성 (dict 매핑 + 키워드 추출)...')
    for r in raw:
        r['lbl_element_code']  = map_element_code(r.get('part_detail',''))
        r['lbl_element_group'] = element_group_of(r['lbl_element_code'])
        r['lbl_part_detail']   = r.get('part_detail') or '기타'
        r['lbl_symptom_code']  = map_symptom_code(
            r.get('defect_description'), r.get('remark'),
            r.get('worker_remark'), r.get('result_remark'))
        r['lbl_cause'] = r.get('cause') or '기타'

    HEADS_DB = ['element_group','element_code','part_detail','symptom_code','cause']

    before_recs = [r for r in raw if norm_phase(r['file_type'])=='BEFORE']
    cls_counts = {h: Counter(r[f'lbl_{h}'] for r in before_recs) for h in HEADS_DB}
    for h in HEADS_DB:
        print(f'  {h:<16}: {len(cls_counts[h])}종')
        print(f'    상위5: {cls_counts[h].most_common(5)}')

    # ── 클래스 확정 (TRAIN_MIN 미만 → 기타) ───────────────────
    print('[3/5] 클래스 확정 (기준 미달 → 기타 통합)...')
    OTHER = '기타'
    main_cls = {h: sorted(v for v,n in cls_counts[h].items()
                          if n >= TRAIN_MIN and v != OTHER)
                for h in HEADS_DB}
    lmaps = {h: {name:i for i,name in enumerate(main_cls[h]+[OTHER])}
             for h in HEADS_DB}
    for h in HEADS_DB:
        print(f'  {h:<16}: {len(main_cls[h])}종 (기준 {TRAIN_MIN}장↑) + 기타')

    # ── metadata.jsonl 생성 ────────────────────────────────────
    print('[4/5] metadata.jsonl 생성...')
    tmp = Path('/content/metadata.jsonl')
    with tmp.open('w', encoding='utf-8') as fp:
        for r in raw:
            phase = norm_phase(r['file_type'])
            is_tr = (phase == 'BEFORE')
            labels = {}
            for h in HEADS_DB:
                v = r[f'lbl_{h}']
                labels[h] = v if v in lmaps[h] else OTHER
            fp.write(json.dumps({
                'image_id':   r['full_path'],
                's3_url':     s3_url(r['full_path']),
                'file_path':  r['full_path'],
                'phase':      phase,
                'area':       r.get('area',''),
                'gongjong':   r.get('gongjong',''),
                'site_code':  r.get('site_code',''),
                'is_trainable': is_tr,
                'split':      assign_split(r['id']) if is_tr else None,
                **{f'lbl_{h}': labels[h] for h in HEADS_DB},
            }, ensure_ascii=False) + '\n')

    print('[5/5] taxonomy.json 생성 + Drive 저장...')
    tax = {
        'version': 'final_v2_realdb',
        'types': TYPES, 'train_min': TRAIN_MIN, 'heads': HEADS_DB,
        'label_maps': lmaps,
        'class_counts': {h: dict(cls_counts[h]) for h in HEADS_DB},
        'note': 'element_group/code는 part_detail 텍스트 dict매핑, symptom은 텍스트 키워드추출, cause는 DB원본',
    }
    json.dump(tax, Path('/content/taxonomy.json').open('w',encoding='utf-8'),
              ensure_ascii=False, indent=2)
    shutil.copy(tmp, META_PATH)
    shutil.copy('/content/taxonomy.json', TAX_PATH)
    recs = [json.loads(l) for l in META_PATH.open(encoding='utf-8')]
    tax  = json.load(TAX_PATH.open(encoding='utf-8'))
    print('✅ 저장 완료')

HEADS_DB = tax['heads']
lmaps    = tax['label_maps']
all_train = [r for r in recs if r.get('is_trainable') and r.get('split')]
print(f'\n전체 학습 가능: {len(all_train):,}장')
print('\n헤드별 클래스 수:')
for h in HEADS_DB:
    print(f'  {h:<16}: {len(lmaps[h])}개')

✅ Drive 캐시 로드 (DB 재조회 생략)

전체 학습 가능: 132,408장

헤드별 클래스 수:
  element_group   : 9개
  element_code    : 27개
  part_detail     : 101개
  symptom_code    : 21개
  cause           : 9개


## 셀 03 — 스테이지별 샘플링 (클래스당 N개 제한)

In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 핵심: element_code 기준으로 클래스당 SAMPLES_PER_CLASS 개 샘플링
# split 비율 유지 (train:val:test = 8:1:1)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

random.seed(SEED)

def stage_sample(recs, samples_per_class, seed=42):
    """
    클래스당 최대 samples_per_class 개로 제한.
    None이면 전체 사용.
    split 비율은 기존 해시 기반 배정 그대로 유지.
    """
    if samples_per_class is None:
        print(f'  샘플 제한 없음 → 전체 {len(recs):,}장 사용')
        return recs

    # element_code 기준 클래스별 그룹핑
    by_cls = defaultdict(list)
    for r in recs:
        by_cls[r.get('lbl_element_code','기타')].append(r)

    sampled = []
    for cls, items in sorted(by_cls.items()):
        random.shuffle(items)
        taken = items[:samples_per_class]
        sampled.extend(taken)

    print(f'  샘플링 결과: {len(sampled):,}장 '
          f'({len(by_cls)}클래스 × 최대 {samples_per_class:,}개)')
    return sampled

train_recs = stage_sample(all_train, SAMPLES_PER_CLASS, SEED)
sp_dist    = Counter(r['split'] for r in train_recs)

print(f'\n  STAGE {STAGE} 학습 데이터:')
print(f'  train: {sp_dist.get("train",0):,}장')
print(f'  val  : {sp_dist.get("val",0):,}장')
print(f'  test : {sp_dist.get("test",0):,}장')
print(f'  합계 : {len(train_recs):,}장')

# 클래스별 분포 확인
cls_dist = Counter(r.get('lbl_element_code') for r in train_recs
                   if r['split']=='train')
print(f'\n  element_code 분포 (상위 10):')
for cls, n in cls_dist.most_common(10):
    bar = '█' * (n // max(1, max(cls_dist.values())//20))
    print(f'    {cls:<25} {n:>5}  {bar}')

# 스테이지 샘플링 정보 저장
stage_info_path = STAGE_DIR / 'stage_info.json'
stage_info = {
    'stage': STAGE,
    'desc':  cfg['desc'],
    'samples_per_class': SAMPLES_PER_CLASS,
    'total': len(train_recs),
    'split': dict(sp_dist),
    'epochs': EPOCHS,
    'batch':  BATCH,
    'created': datetime.datetime.now().isoformat(),
}
stage_info_path.write_text(json.dumps(stage_info, ensure_ascii=False, indent=2))

  샘플링 결과: 2,610장 (27클래스 × 최대 100개)

  STAGE 1 학습 데이터:
  train: 2,071장
  val  : 259장
  test : 280장
  합계 : 2,610장

  element_code 분포 (상위 10):
    SAN.SIN                      88  ██████████████████████
    MEP.OUT                      86  █████████████████████
    MEP.SWT                      85  █████████████████████
    WIN.HDW                      85  █████████████████████
    WIN.WDW                      84  █████████████████████
    FIN.FLR                      83  ████████████████████
    FIN.WLP                      83  ████████████████████
    FIN.PNT                      82  ████████████████████
    SAN.BTH                      82  ████████████████████
    DOR.SLD                      81  ████████████████████


230

## 셀 04 — S3 → Drive 다운로드 (이어받기)

In [8]:
# ✅ 전체 이미지를 Drive에 한 번만 저장
# ✅ 이미 있는 파일은 건너뜀 → 스테이지 변경해도 재다운로드 불필요

# Colab 로컬 임시 저장 경로
LOCAL_TEMP_IMG_DIR = Path('/content/ds_images_temp')
LOCAL_TEMP_IMG_DIR.mkdir(parents=True, exist_ok=True) # Ensure temp directory exists

def load_prog():
    if DL_PROG.exists():
        try:
            p = json.loads(DL_PROG.read_text())
            print(f'✅ 다운로드 진행 로드: 완료 {p["done"]:,} 실패 {p["failed"]:,}')
            return p
        except Exception: pass
    return {'done':0,'failed':0,'done_ids':[],'failed_ids':[]}

def save_prog(p):
    p['ts'] = datetime.datetime.now().isoformat()
    DL_PROG.write_text(json.dumps(p, ensure_ascii=False))

def dl_file(url, dst, retries=3):
    # dst는 이제 로컬 임시 경로
    if Path(dst).exists() and Path(dst).stat().st_size > 0:
        return True
    for i in range(1, retries+1):
        try:
            urllib.request.urlretrieve(url, dst)
            if Path(dst).stat().st_size > 0: return True
        except Exception:
            if i < retries: time.sleep(i)
    return False

prog     = load_prog()
done_set = set(str(x) for x in prog['done_ids'])
fail_set = set(str(x) for x in prog['failed_ids'])

# 현 스테이지 샘플만 다운로드 (전체 대신 현재 필요한 것만)
seen, deduped = {}, []
for r in train_recs:
    iid = r['image_id']
    if iid not in seen:
        seen[iid] = True
        deduped.append(r)

# `need_dl`은 아직 다운로드되지 않은 (`done_set`에 없는) 이미지 목록
need_dl = [r for r in deduped if str(r['image_id']) not in done_set]
print(f'STAGE {STAGE}: {len(deduped):,}장 필요 | 이미 완료: {len(done_set):,} | 다운로드 필요: {len(need_dl):,}')

if len(need_dl) == 0:
    print('✅ 이미 모두 다운로드됨 → 셀 05로 이동')
else:
    # 다운로드 상태를 추적하기 위한 변수
    actual_downloads_made = False

    pbar = tqdm(deduped, total=len(deduped), initial=len(done_set),
                desc=f'S3→LocalTemp(Stage{STAGE})', unit='파일', dynamic_ncols=True)
    batch_n = 0
    for r in pbar:
        iid = str(r['image_id'])
        if iid in done_set:
            # 이미 다운로드 완료 목록에 있는 파일은 건너뜀
            # 이 시점에서 파일이 Google Drive에 있다고 가정함
            pbar.set_postfix(done=prog['done'], fail=prog['failed'], refresh=False)
            continue

        grp = safe(r.get('lbl_element_group','unknown'))
        cls = safe(r.get('lbl_element_code','unknown'))
        fn  = safe(Path(r['file_path']).name)
        # 로컬 임시 경로를 사용
        dst = LOCAL_TEMP_IMG_DIR / r['split'] / grp / cls / fn
        dst.parent.mkdir(parents=True, exist_ok=True)

        # dl_file 함수는 로컬 경로에 파일이 이미 있는지 확인하고 없으면 다운로드
        if dst.exists() and dst.stat().st_size > 0: # 로컬에 파일이 있으면 다운로드 스킵
             # 로컬에 있지만 done_set에 없는 경우는 처음 다운로드되었고 prog가 업데이트 안된 경우 (e.g., Ctrl+C)
             # 이 경우도 실제 다운로드는 없었으니 actual_downloads_made는 False 유지
            prog['done'] += 1; done_set.add(iid); prog['done_ids'].append(iid)
        else: # 로컬에도 파일이 없으면 S3에서 다운로드 시도
            ok = dl_file(r.get('s3_url',''), str(dst))
            if ok:
                actual_downloads_made = True # 실제 다운로드가 발생했음
                prog['done'] += 1; done_set.add(iid); prog['done_ids'].append(iid)
                if iid in fail_set:
                    fail_set.discard(iid)
                    prog['failed_ids'] = [x for x in prog['failed_ids'] if str(x)!=iid]
                    prog['failed'] = len(prog['failed_ids'])
            else:
                if iid not in fail_set:
                    prog['failed'] += 1; fail_set.add(iid); prog['failed_ids'].append(iid)

        pbar.set_postfix(done=prog['done'], fail=prog['failed'], refresh=False)
        batch_n += 1
        if batch_n % BATCH_DL == 0:
            save_prog(prog)
            tqdm.write(f'  💾 체크포인트: {prog["done"]:,}건 완료')

    pbar.close()
    save_prog(prog)

    # 실패 재시도 (로컬 임시 경로로 다운로드 시도)
    if prog['failed_ids']:
        print(f'\n실패 {prog["failed"]}건 재시도...')
        retry = [r for r in deduped
                 if str(r['image_id']) in set(str(x) for x in prog['failed_ids'])]
        prog['failed_ids'] = []; prog['failed'] = 0; fail_set.clear()
        for r in tqdm(retry, desc='재시도'):
            iid = str(r['image_id'])
            grp = safe(r.get('lbl_element_group','unknown'))
            cls = safe(r.get('lbl_element_code','unknown'))
            dst = LOCAL_TEMP_IMG_DIR / r['split'] / grp / cls / safe(Path(r['file_path']).name)
            dst.parent.mkdir(parents=True, exist_ok=True)
            ok = dl_file(r.get('s3_url',''), str(dst)) # 로컬 임시 경로로 다운로드
            if ok:
                actual_downloads_made = True # 재시도에서도 실제 다운로드가 발생했음
                prog['done'] += 1; done_set.add(iid); prog['done_ids'].append(iid)
            else:
                prog['failed'] += 1; prog['failed_ids'].append(iid)
        save_prog(prog)

    # 실제 다운로드된 파일이 있다면 Google Drive로 복사
    if actual_downloads_made:
        print('\n모든 이미지 다운로드 완료 (로컬 임시 저장소). Google Drive로 복사 중...')
        # IMG_DIR (전역 변수)은 /content/drive/MyDrive/AI_Checker_Final/ds_images 를 가리킴
        # LOCAL_TEMP_IMG_DIR의 하위 폴더 (train, val, test)들을 IMG_DIR 아래로 복사
        for src_split_dir in LOCAL_TEMP_IMG_DIR.iterdir():
            if src_split_dir.is_dir():
                dest_split_dir = IMG_DIR / src_split_dir.name
                shutil.copytree(src_split_dir, dest_split_dir, dirs_exist_ok=True)
        print('✅ Google Drive 복사 완료')
    else:
        print('\n새롭게 다운로드된 이미지가 없어 Google Drive로의 복사는 건너뜠니다.')


print('\n다운로드 완료:')
for sp in ['train','val','test']:
    # 최종적으로 Google Drive 경로에서 파일 수를 세도록 수정
    n_jpg = len(list((IMG_DIR/sp).rglob('*.jpg')))
    n_png = len(list((IMG_DIR/sp).rglob('*.png')))
    n = n_jpg + n_png
    print(f'  {"✅" if n>0 else "❌"} {sp}: {n:,}장')

✅ 다운로드 진행 로드: 완료 2,609 실패 1
STAGE 1: 2,610장 필요 | 이미 완료: 2,609 | 다운로드 필요: 1


S3→LocalTemp(Stage1): 100%|#########9| 2609/2610 [00:00<?, ?파일/s]


실패 1건 재시도...


재시도:   0%|          | 0/1 [00:00<?, ?it/s]


새롭게 다운로드된 이미지가 없어 Google Drive로의 복사는 건너뜠니다.

다운로드 완료:
  ✅ train: 2,003장
  ✅ val: 254장
  ✅ test: 275장


## 셀 05 — 모델 + 클래스 가중치

In [9]:
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEV}')

MEAN = [0.485,0.456,0.406]; STD = [0.229,0.224,0.225]

area_map = {}; gj_map = {}
for r in train_recs:
    a = r.get('area',''); g = r.get('gongjong','')
    if a not in area_map: area_map[a] = len(area_map)
    if g not in gj_map:   gj_map[g]   = len(gj_map)

def make_weights(recs, hk, lmap, boost=1.0):
    counts = Counter(r.get(f'lbl_{hk}') for r in recs if r.get('split')=='train')
    n = len(lmap); total = sum(counts.values()); mean_c = total/max(n,1)
    w = torch.ones(n)
    for name, idx in lmap.items():
        c = max(counts.get(name,1),1)
        w[idx] = total/(n*c)
        if c < mean_c: w[idx] *= boost
    return w/w.mean()

weights = {h: make_weights(train_recs, h, lmaps[h],
                            MINORITY_BOOST if h in ['part_detail','symptom_code'] else 1.0)
           for h in HEADS_DB}

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMGSZ, scale=(.70,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=.3, contrast=.3, saturation=.2),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD),
])
val_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMGSZ),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD),
])

class DefectDS(Dataset):
    def __init__(self, recs, tf=None):
        self.recs = recs; self.tf = tf
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r   = self.recs[i]
        grp = safe(r.get('lbl_element_group','unknown'))
        cls = safe(r.get('lbl_element_code','unknown'))
        fp  = IMG_DIR/r['split']/grp/cls/safe(Path(r['file_path']).name)
        try:    img = Image.open(fp).convert('RGB')
        except: img = Image.new('RGB',(IMGSZ,IMGSZ),(128,128,128))
        if self.tf: img = self.tf(img)
        lbl = {h: lmaps[h].get(r.get(f'lbl_{h}',''),0) for h in HEADS_DB}
        return img, area_map.get(r.get('area',''),0), gj_map.get(r.get('gongjong',''),0), lbl

splits_data = {s:[r for r in train_recs if r['split']==s] for s in ['train','val','test']}
train_ld = DataLoader(DefectDS(splits_data['train'],train_tf), batch_size=BATCH,
                      shuffle=True, num_workers=2, pin_memory=(DEV=='cuda'))
val_ld   = DataLoader(DefectDS(splits_data['val'],val_tf), batch_size=BATCH,
                      shuffle=False, num_workers=2, pin_memory=(DEV=='cuda'))

print(f'train {len(splits_data["train"])} | val {len(splits_data["val"])} | test {len(splits_data["test"])}')

class FiveLayerModel(nn.Module):
    def __init__(self, n_area, n_gj, head_sizes, area_dim=16, gj_dim=8):
        super().__init__()
        self.bb = timm.create_model(MODEL_N, pretrained=True, num_classes=0)
        fd = self.bb.num_features
        self.ae = nn.Embedding(n_area+1, area_dim)
        self.ge = nn.Embedding(n_gj+1, gj_dim)
        in_d = fd+area_dim+gj_dim
        self.heads = nn.ModuleDict({
            h: nn.Sequential(
                nn.LayerNorm(in_d), nn.Linear(in_d,512), nn.GELU(), nn.Dropout(.3),
                nn.Linear(512,256), nn.GELU(), nn.Dropout(.2), nn.Linear(256,n),
            ) for h,n in head_sizes.items()
        })
    def forward(self, img, ai, gi):
        x = torch.cat([self.bb(img), self.ae(ai), self.ge(gi)], dim=1)
        return {h: head(x) for h,head in self.heads.items()}

head_sizes = {h: len(lmaps[h]) for h in HEADS_DB}
model = FiveLayerModel(len(area_map), len(gj_map), head_sizes).to(DEV)
crits = {h: nn.CrossEntropyLoss(weight=weights[h].to(DEV)) for h in HEADS_DB}
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

print(f'모델: {MODEL_N} | 파라미터: {sum(p.numel() for p in model.parameters()):,}')

device: cuda
train 2071 | val 259 | test 280


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

모델: convnext_tiny | 파라미터: 30,558,695


## 셀 06 — 학습 루프 (이어받기 + 체크포인트 + Early Stopping)

In [10]:
start_epoch = 1; best_acc = 0.0; no_improve = 0; log_rows = []

if LAST_PATH.exists():
    ck = torch.load(LAST_PATH, map_location=DEV)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    sched.load_state_dict(ck['sched_state'])
    start_epoch = ck['epoch']+1
    best_acc    = ck.get('best_acc',0.0)
    no_improve  = ck.get('no_improve',0)
    log_rows    = json.loads(LOG_PATH.read_text()) if LOG_PATH.exists() else []
    print(f'✅ 이전 체크포인트 복원 → epoch {ck["epoch"]}부터 재개 (best={best_acc:.4f})')
else:
    print(f'STAGE {STAGE} 새 학습 시작 — {cfg["desc"]}')

HS = {'element_group':'grp','element_code':'elem',
      'part_detail':'part','symptom_code':'symp','cause':'caus'}

print(f'\n{"Ep":>3} {"Loss":>8} '
      +' '.join(f'{HS[h]:>6}' for h in HEADS_DB)
      +f' {"Avg":>7} {"sec":>5}')
print('─'*58)

for epoch in range(start_epoch, EPOCHS+1):
    t0 = time.time()
    model.train()
    tloss = tcnt = 0
    for imgs,ai,gi,lbl in train_ld:
        imgs=imgs.to(DEV); ai=ai.to(DEV); gi=gi.to(DEV)
        labs={h:lbl[h].to(DEV) for h in HEADS_DB}
        opt.zero_grad()
        out = model(imgs,ai,gi)
        loss= sum(HEAD_LOSS_W[h]*crits[h](out[h],labs[h]) for h in HEADS_DB)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        tloss+=loss.item()*imgs.size(0); tcnt+=imgs.size(0)

    model.eval(); accs={h:0 for h in HEADS_DB}; vn=0
    with torch.no_grad():
        for imgs,ai,gi,lbl in val_ld:
            imgs=imgs.to(DEV); ai=ai.to(DEV); gi=gi.to(DEV)
            labs={h:lbl[h].to(DEV) for h in HEADS_DB}
            out=model(imgs,ai,gi)
            for h in HEADS_DB: accs[h]+=(out[h].argmax(1)==labs[h]).sum().item()
            vn+=imgs.size(0)
    sched.step()

    avg_acc = sum(accs[h] for h in HEADS_DB)/(len(HEADS_DB)*max(vn,1))
    sec = time.time()-t0
    flag = ' ★' if avg_acc>best_acc else ''

    print(f'{epoch:>3} {tloss/tcnt:>8.4f} '
          +' '.join(f'{accs[h]/max(vn,1):>6.3f}' for h in HEADS_DB)
          +f' {avg_acc:>7.4f} {sec:>4.0f}s{flag}')

    if avg_acc > best_acc:
        best_acc=avg_acc; no_improve=0
        torch.save({'epoch':epoch,'val_acc':avg_acc,
                    'model_state':model.state_dict(),
                    'head_sizes':head_sizes,'lmaps':lmaps,
                    'area_map':area_map,'gj_map':gj_map,
                    'model_name':MODEL_N,'img_size':IMGSZ,'heads':HEADS_DB,
                    'stage':STAGE,'samples_per_class':SAMPLES_PER_CLASS,
                   }, BEST_PATH)
        print(f'  💾 best.pth (val_acc={avg_acc:.4f})')
    else:
        no_improve+=1

    torch.save({'epoch':epoch,'best_acc':best_acc,'no_improve':no_improve,
                'model_state':model.state_dict(),
                'opt_state':opt.state_dict(),'sched_state':sched.state_dict(),
               }, LAST_PATH)

    log_rows.append({'epoch':epoch,'loss':round(tloss/tcnt,6),
                     'avg_acc':round(avg_acc,6),
                     **{f'{HS[h]}_acc':round(accs[h]/max(vn,1),6) for h in HEADS_DB}})
    LOG_PATH.write_text(json.dumps(log_rows,indent=2))

    if no_improve >= PATIENCE:
        print(f'\n⏹️  Early Stopping (epoch {epoch}, patience={PATIENCE})')
        break

print(f'\n✅ STAGE {STAGE} 학습 완료')
print(f'   Best Val Acc: {best_acc:.4f} ({best_acc*100:.1f}%)')
print(f'   저장: {BEST_PATH}')

STAGE 1 새 학습 시작 — 스모크 테스트 (15분)

 Ep     Loss    grp   elem   part   symp   caus     Avg   sec
──────────────────────────────────────────────────────────
  1  25.5232  0.143  0.050  0.000  0.023  0.085  0.0602 1405s ★
  💾 best.pth (val_acc=0.0602)
  2  25.0531  0.297  0.097  0.019  0.031  0.506  0.1900  195s ★
  💾 best.pth (val_acc=0.1900)
  3  24.3670  0.417  0.371  0.015  0.031  0.201  0.2069  207s ★
  💾 best.pth (val_acc=0.2069)

✅ STAGE 1 학습 완료
   Best Val Acc: 0.2069 (20.7%)
   저장: /content/drive/MyDrive/AI_Checker_Final/stage_1/checkpoints/best.pth


## 셀 07 — 테스트 평가 + 스테이지 비교 리포트

In [11]:
test_ld = DataLoader(DefectDS(splits_data['test'],val_tf),
                     batch_size=BATCH,shuffle=False,num_workers=2)
ck = torch.load(BEST_PATH,map_location=DEV)
mdl = FiveLayerModel(len(area_map),len(gj_map),head_sizes).to(DEV)
mdl.load_state_dict(ck['model_state']); mdl.eval()

y_true={h:[] for h in HEADS_DB}; y_pred={h:[] for h in HEADS_DB}
with torch.no_grad():
    for imgs,ai,gi,lbl in tqdm(test_ld,desc='테스트'):
        imgs=imgs.to(DEV); ai=ai.to(DEV); gi=gi.to(DEV)
        out=mdl(imgs,ai,gi)
        for h in HEADS_DB:
            y_true[h].extend(lbl[h].numpy())
            y_pred[h].extend(out[h].argmax(1).cpu().numpy())

print(f'\n{"="*56}')
print(f'  STAGE {STAGE} 테스트 결과 — {cfg["desc"]}')
print(f'  샘플: {SAMPLES_PER_CLASS or "전체"}장/클래스')
print(f'{"="*56}')
test_accs = {}
for h in HEADS_DB:
    acc = (np.array(y_true[h])==np.array(y_pred[h])).mean()
    test_accs[h] = round(float(acc),4)
    bar = '█'*int(acc*20)
    print(f'  {h:<16} {acc*100:>6.1f}%  {bar}')

# 스테이지 비교 리포트 (누적)
report_path = Path(f'{BK}/stage_comparison.json')
if report_path.exists():
    report = json.loads(report_path.read_text())
else:
    report = {}

report[f'stage_{STAGE}'] = {
    'stage': STAGE,
    'desc':  cfg['desc'],
    'samples_per_class': SAMPLES_PER_CLASS,
    'total_train': len(splits_data['train']),
    'best_val_acc': round(best_acc,4),
    'test_accs': test_accs,
    'avg_test_acc': round(float(np.mean(list(test_accs.values()))),4),
    'epochs_run': len(log_rows),
    'created': datetime.datetime.now().isoformat(),
}
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2))

# 전체 스테이지 비교 출력
print(f'\n{"─"*56}')
print('  전체 스테이지 비교:')
print(f'  {"Stage":<8} {"샘플/cls":>10} {"Val Acc":>9} {"Test Avg":>9}')
print(f'  {"─"*40}')
for k in sorted(report.keys()):
    v = report[k]
    spc = f'{v["samples_per_class"]:,}' if v["samples_per_class"] else '전체'
    print(f'  {k:<8} {spc:>10} {v["best_val_acc"]*100:>8.1f}% {v["avg_test_acc"]*100:>8.1f}%')

next_stage = STAGE + 1
if next_stage <= 5:
    ns = STAGE_CONFIG[next_stage]
    print(f'\n→ 다음: 셀 01에서 STAGE = {next_stage} 으로 변경')
    print(f'  ({ns["desc"]} / {ns["samples_per_class"] or "전체"}장/클래스)')
else:
    print(f'\n✅ 모든 스테이지 완료 → 셀 08 (온디바이스 변환) 실행')

테스트:   0%|          | 0/9 [00:00<?, ?it/s]


  STAGE 1 테스트 결과 — 스모크 테스트 (15분)
  샘플: 100장/클래스
  element_group      38.6%  ███████
  element_code       37.5%  ███████
  part_detail         1.8%  
  symptom_code        1.8%  
  cause              16.1%  ███

────────────────────────────────────────────────────────
  전체 스테이지 비교:
  Stage        샘플/cls   Val Acc  Test Avg
  ────────────────────────────────────────
  stage_1         100     20.7%     19.1%

→ 다음: 셀 01에서 STAGE = 2 으로 변경
  (빠른 검증 (30분) / 300장/클래스)


## 셀 08 — 온디바이스 변환 (best.pth → ONNX → TFLite)

In [22]:
!pip uninstall -y onnx onnxruntime onnxscript 2>/dev/null
!pip install -q -U onnx onnxruntime onnxscript ai-edge-torch 2>/dev/null
import onnx

ONNX_PATH   = EXPORT_DIR / f'model_stage{STAGE}.onnx'
TFLITE_PATH = EXPORT_DIR / f'model_stage{STAGE}_int8.tflite'
META_OUT    = EXPORT_DIR / f'model_meta_stage{STAGE}.json'

class ExportWrapper(nn.Module):
    def __init__(self, base):
        super().__init__(); self.m = base
    def forward(self, img, ai, gi):
        out = self.m(img,ai,gi)
        return tuple(torch.softmax(out[h],dim=1) for h in HEADS_DB)

ck   = torch.load(BEST_PATH, map_location='cpu')
base = FiveLayerModel(len(area_map),len(gj_map),head_sizes)
base.load_state_dict(ck['model_state']); base.eval()
wrapper = ExportWrapper(base).eval()

dummy_img = torch.zeros(1,3,IMGSZ,IMGSZ)
dummy_ai  = torch.zeros(1,dtype=torch.long)
dummy_gi  = torch.zeros(1,dtype=torch.long)

print(f'STAGE {STAGE} 모델 변환 중...')
with torch.no_grad():
    torch.onnx.export(wrapper,(dummy_img,dummy_ai,dummy_gi),
                      str(ONNX_PATH),opset_version=14,
                      input_names=['image','area_idx','gj_idx'],
                      output_names=HEADS_DB,
                      do_constant_folding=True,
                      verbose=False,
                      dynamo=False)
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
print(f'✅ ONNX: {ONNX_PATH.name} ({ONNX_PATH.stat().st_size/1024/1024:.1f} MB)')

try:
    import ai_edge_torch
    edge = ai_edge_torch.convert(wrapper,(dummy_img,dummy_ai,dummy_gi))
    edge.export(str(TFLITE_PATH))
    print(f'✅ TFLite: {TFLITE_PATH.name} ({TFLITE_PATH.stat().st_size/1024/1024:.1f} MB)')
except Exception as e:
    print(f'ai_edge_torch 실패({e}) → ONNX INT8 사용')
    from onnxruntime.quantization import quantize_dynamic, QuantType
    TFLITE_PATH = EXPORT_DIR / f'model_stage{STAGE}_int8.onnx'
    quantize_dynamic(str(ONNX_PATH),str(TFLITE_PATH),weight_type=QuantType.QInt8)
    print(f'✅ ONNX INT8: {TFLITE_PATH.name} ({TFLITE_PATH.stat().st_size/1024/1024:.1f} MB)')

meta = {'version':f'stage{STAGE}','model_name':MODEL_N,'img_size':IMGSZ,
        'stage':STAGE,'samples_per_class':SAMPLES_PER_CLASS,
        'val_acc':round(float(ck.get('val_acc',0)),4),
        'heads':HEADS_DB,
        'input':{'image':[1,3,IMGSZ,IMGSZ],'area_idx':[1],'gj_idx':[1]},
        'label_maps':{h:{str(v):k for k,v in lmaps[h].items()} for h in HEADS_DB},
        'area_map':area_map,'gj_map':gj_map}
META_OUT.write_text(json.dumps(meta,ensure_ascii=False,indent=2))

print(f'\n✅ STAGE {STAGE} 변환 완료')
print(f'\nAndroid 앱 assets/ 에 복사:')
print(f'  {TFLITE_PATH.name}')
print(f'  {META_OUT.name}')
print(f'\n저장 경로: {EXPORT_DIR}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.2 MB/s eta 0:00:00
STAGE 1 모델 변환 중...


/tmp/ipykernel_624/2937990552.py:27: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrapper,(dummy_img,dummy_ai,dummy_gi),


✅ ONNX: model_stage1.onnx (116.7 MB)
ai_edge_torch 실패(module 'ai_edge_torch' has no attribute 'convert') → ONNX INT8 사용


/tmp/ipykernel_624/2937990552.py:38: DeprecationWarning: 

'ai-edge-torch' is deprecated and has been renamed to 'litert-torch'. Please update your dependencies and imports. The 'ai-edge-torch' package will not receive further updates.

  import ai_edge_torch


✅ ONNX INT8: model_stage1_int8.onnx (29.7 MB)

✅ STAGE 1 변환 완료

Android 앱 assets/ 에 복사:
  model_stage1_int8.onnx
  model_meta_stage1.json

저장 경로: /content/drive/MyDrive/AI_Checker_Final/stage_1/export
